In [1]:
from typing import Any, Dict

import hydra
import numpy as np
import omegaconf
import torch
import pytorch_lightning as pl
import torch.nn as nn
from torch.nn import functional as F
from torch_scatter import scatter
from tqdm import tqdm

from cdvae.common.utils import PROJECT_ROOT
from cdvae.common.data_utils import (
    EPSILON, cart_to_frac_coords, mard, lengths_angles_to_volume,
    frac_to_cart_coords, min_distance_sqr_pbc)
from cdvae.pl_modules.embeddings import MAX_ATOMIC_NUM
from cdvae.pl_modules.embeddings import KHOT_EMBEDDINGS

#added by Tsach
from pymatgen.core.structure import Structure
from pymatgen.core.periodic_table import Element
from pymatgen.core.lattice import Lattice
from pymatgen.analysis.diffraction.xrd import XRDCalculator
#import Batch
from torch_geometric.data import Batch
xrd_calculator = XRDCalculator(wavelength='CuKa', symprec=0.1)
torch.set_printoptions(threshold=50000) # use this if you want to print the entire tensor


import time
import argparse
import torch

from tqdm import tqdm
from torch.optim import Adam
from pathlib import Path
from types import SimpleNamespace
from torch_geometric.data import Batch
from torch_geometric.data.separate import separate

#import a library that allows you to reload a module
from importlib import reload

from eval_utils import load_model

all_frac_coords_stack = []
all_atom_types_stack = []
frac_coords = []
num_atoms = []
atom_types = []
lengths = []
angles = []
input_data_list = []

#my code 
list_of_idxs = []
list_of_batchs = []

import importlib 
import eval_utils
importlib.reload(eval_utils)
from eval_utils import load_model
model_path = Path("/home/gridsan/tmackey/hydra/singlerun/2023-10-29/no_encode_intensity_concat_comp_concat_neg_mask_mp_20/")
model, test_loader, cfg = load_model(model_path, True)

loader = test_loader

/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/hydra/experimental/compose.py:16: UserWarning: hydra.experimental.compose() is no longer experimental. Use hydra.compose()
  warnings.warn(


  0%|          | 0/9046 [00:00<?, ?it/s]

/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/pymatgen/io/cif.py:1168: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/pymatgen/io/cif.py:1168: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/pymatgen/io/cif.py:1168: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
/home/gridsan/tmackey/.conda/envs/cdvae/

CrystDataModule(self.datasets={'train': {'_target_': 'cdvae.pl_data.dataset.CrystDataset', 'name': 'Formation energy train', 'path': '/home/gridsan/tmackey/cdvae/data/mp_20/train.csv', 'prop': 'formation_energy_per_atom', 'niggli': True, 'primitive': False, 'graph_method': 'crystalnn', 'lattice_scale_method': 'scale_length', 'preprocess_workers': 30}, 'val': [{'_target_': 'cdvae.pl_data.dataset.CrystDataset', 'name': 'Formation energy val', 'path': '/home/gridsan/tmackey/cdvae/data/mp_20/val.csv', 'prop': 'formation_energy_per_atom', 'niggli': True, 'primitive': False, 'graph_method': 'crystalnn', 'lattice_scale_method': 'scale_length', 'preprocess_workers': 30}], 'test': [{'_target_': 'cdvae.pl_data.dataset.CrystDataset', 'name': 'Formation energy test', 'path': '/home/gridsan/tmackey/cdvae/data/mp_20/test.csv', 'prop': 'formation_energy_per_atom', 'niggli': True, 'primitive': False, 'graph_method': 'crystalnn', 'lattice_scale_method': 'scale_length', 'preprocess_workers': 30}]}, self

/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/torch_geometric/deprecation.py:13: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


In [2]:
model.to('cuda:0')

CDVAE(
  (encoder): DimeNetPlusPlusWrap(
    (rbf): BesselBasisLayer(
      (envelope): Envelope()
    )
    (sbf): SphericalBasisLayer(
      (envelope): Envelope()
    )
    (emb): EmbeddingBlock(
      (emb): Embedding(95, 128)
      (lin_rbf): Linear(in_features=6, out_features=128, bias=True)
      (lin): Linear(in_features=384, out_features=128, bias=True)
    )
    (output_blocks): ModuleList(
      (0): OutputPPBlock(
        (lin_rbf): Linear(in_features=6, out_features=128, bias=False)
        (lin_up): Linear(in_features=128, out_features=256, bias=True)
        (lins): ModuleList(
          (0): Linear(in_features=256, out_features=256, bias=True)
          (1): Linear(in_features=256, out_features=256, bias=True)
          (2): Linear(in_features=256, out_features=256, bias=True)
        )
        (lin): Linear(in_features=256, out_features=256, bias=False)
      )
      (1): OutputPPBlock(
        (lin_rbf): Linear(in_features=6, out_features=128, bias=False)
        (lin

In [3]:
for idx, batch in enumerate(loader):
    print(idx)
    list_of_idxs.append(idx)
    list_of_batchs.append(batch)

idx = list_of_idxs[0]
batch = list_of_batchs[0]

def new_dataloader_batch_processor(batch): 
    batch_reserve = batch
    xrd_int = batch_reserve[1]
    xrd_loc = batch_reserve[2]
    atom_spec = batch_reserve[3]
    batch = batch[0]
    return batch_reserve, xrd_int, xrd_loc, atom_spec, batch

batch_reserve, xrd_int, xrd_loc, atom_spec, batch = new_dataloader_batch_processor(batch)
all_frac_coords_stack = []
all_atom_types_stack = []
frac_coords = []
num_atoms = []
atom_types = []
lengths = []
angles = []
input_data_list = []

batch_all_frac_coords = []
batch_all_atom_types = []
batch_frac_coords, batch_num_atoms, batch_atom_types = [], [], []
batch_lengths, batch_angles = [], []

#wrecking the batch value to demonstrate decoding only from the xrd info
_, _, z = model.encode(None, xrd_int, xrd_loc, atom_spec)

#take only the first crystal to avoid issues with memory 
z = z[[range(1)]]

self = model

num_atoms = batch.num_atoms[[range(1)]]
num_atoms = num_atoms.to('cuda:0')

atom_spec = atom_spec[[range(1)]]
atom_spec = atom_spec.to('cuda:0')

z = z.to('cuda:0')

force_num_atoms = True
gt_num_atoms = num_atoms if force_num_atoms else None

gt_atom_types = None

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35


I noticed some sketchy behavior during the diffusion process before, with one of the worse models. Especially in regards to atomic composition, it seemed to push the atom types towards a smaller number of atoms. I wanted to check this out in the notebook. 
Specifically, the questions I wanted to answer are: 
* Does diffusion make predictions worse or better?
* Might make sense to break down the investigation into four parts: 
    * Validity: what is the validity at each step? 
    * match rates / coordinate rmse (note that this is dependent on validity): is the crystal matched at each step?
    * Composition: what is the composition at each step? 


In [4]:
#number of steps set unusually low to get a rough estimate of performance
ld_kwargs = SimpleNamespace(n_step_each=10,
                                step_lr=1e-4,
                                min_sigma=0,
                                save_traj=False,
                                disable_bar=False)

In [5]:
batch.atom_types[[range(gt_num_atoms)]]

tensor([31, 31, 31, 31, 52, 52, 52, 52])

In [6]:
#first step: create a ground truth crystal for comparison 

#get the lattice parameters and atom types and fractional coorinates for the first crystal 
gt_lengths = batch.lengths[[0]]
gt_angles = batch.angles[[0]]
gt_frac_coords = batch.frac_coords[[range(gt_num_atoms)]]
gt_types = batch.atom_types[[range(gt_num_atoms)]]

In [7]:
lengths = gt_lengths.numpy().flatten()
angles = gt_angles.numpy().flatten()

# Initialize the Lattice
lattice = Lattice.from_parameters(lengths[0], lengths[1], lengths[2], angles[0], angles[1], angles[2])

In [8]:
gt_tructure = Structure(lattice, species=gt_types, coords=gt_frac_coords, coords_are_cartesian=False)

In [9]:
gt_tructure

Structure Summary
Lattice
    abc : 4.134600162506107 4.134599673861435 18.425569534301758
 angles : 90.00000250447802 90.00000250447812 120.00000390950429
 volume : 272.78376468258256
      A : 4.1346001625061035 0.0 -1.8072911700528493e-07
      B : -2.0673000812530518 3.5806682109832764 -1.807290885835755e-07
      C : 0.0 0.0 18.425569534301758
    pbc : True True True
PeriodicSite: Ga (2.067, 1.194, 15.05) [0.6667, 0.3333, 0.817]
PeriodicSite: Ga (0.0, 2.387, 5.842) [0.3333, 0.6667, 0.317]
PeriodicSite: Ga (0.0, 2.387, 3.371) [0.3333, 0.6667, 0.183]
PeriodicSite: Ga (2.067, 1.194, 12.58) [0.6667, 0.3333, 0.683]
PeriodicSite: Te (2.067, 1.194, 2.097) [0.6667, 0.3333, 0.1138]
PeriodicSite: Te (0.0, 2.387, 11.31) [0.3333, 0.6667, 0.6138]
PeriodicSite: Te (0.0, 2.387, 16.33) [0.3333, 0.6667, 0.8862]
PeriodicSite: Te (2.067, 1.194, 7.116) [0.6667, 0.3333, 0.3862]

In [10]:
gt_elements_input=atom_spec

In [11]:
if ld_kwargs.save_traj:
    all_frac_coords = []
    all_pred_cart_coord_diff = []
    all_noise_cart = []
    all_atom_types = []

# obtain key stats.
if not self.use_composition_constraint: 
    gt_elements_input = None

num_atoms, _, lengths, angles, composition_per_atom = self.decode_stats(
    z, gt_num_atoms, gt_elements=gt_elements_input)

if gt_num_atoms is not None:
    num_atoms = gt_num_atoms

composition_per_atom = F.softmax(composition_per_atom, dim=-1)

if gt_atom_types is None:
    composition_per_atom = composition_per_atom.cuda(0)
    num_atoms = num_atoms.cuda(0)
    cur_atom_types = self.sample_composition(
        composition_per_atom, num_atoms)
else:
    cur_atom_types = gt_atom_types

# init coords.
cur_frac_coords = torch.rand((num_atoms.sum(), 3), device=z.device)

/home/gridsan/tmackey/cdvae/cdvae/common/data_utils.py:625: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float)


In [12]:
cur_atom_types

tensor([52, 52, 52, 31, 31, 52, 31, 52], device='cuda:0')

In [13]:
cur_frac_coords

tensor([[0.4420, 0.3819, 0.2054],
        [0.1870, 0.0582, 0.8044],
        [0.4016, 0.9496, 0.8518],
        [0.9396, 0.5559, 0.5847],
        [0.4776, 0.4777, 0.8853],
        [0.9659, 0.9637, 0.0353],
        [0.7786, 0.1875, 0.8331],
        [0.2367, 0.1603, 0.4752]], device='cuda:0')

In [14]:
class Crystal(object):
    def __init__(self, crys_array_dict):
        self.frac_coords = crys_array_dict['frac_coords']
        self.atom_types = crys_array_dict['atom_types']
        self.lengths = crys_array_dict['lengths']
        self.angles = crys_array_dict['angles']
        self.dict = crys_array_dict

        self.get_structure()
        self.get_composition()
        self.get_validity()
        self.get_fingerprints()
    def get_structure(self):
        if min(self.lengths.tolist()) < 0:
            self.constructed = False
            self.invalid_reason = 'non_positive_lattice'
        else:
            try:
                self.structure = Structure(
                    lattice=Lattice.from_parameters(
                        *(self.lengths.tolist() + self.angles.tolist())),
                    species=self.atom_types, coords=self.frac_coords, coords_are_cartesian=False)
                self.constructed = True
            except Exception:
                self.constructed = False
                self.invalid_reason = 'construction_raises_exception'
            if self.structure.volume < 0.1:
                self.constructed = False
                self.invalid_reason = 'unrealistically_small_lattice'
    def get_composition(self):
        elem_counter = Counter(self.atom_types)
        composition = [(elem, elem_counter[elem])
                       for elem in sorted(elem_counter.keys())]
        elems, counts = list(zip(*composition))
        counts = np.array(counts)
        counts = counts / np.gcd.reduce(counts)
        self.elems = elems
        self.comps = tuple(counts.astype('int').tolist())
    def get_validity(self):
        self.comp_valid = smact_validity(self.elems, self.comps)
        if self.constructed:
            self.struct_valid = structure_validity(self.structure)
        else:
            self.struct_valid = False
        self.valid = self.comp_valid and self.struct_valid
    def get_fingerprints(self):
        elem_counter = Counter(self.atom_types)
        comp = Composition(elem_counter)
        self.comp_fp = CompFP.featurize(comp)
        try:
            site_fps = [CrystalNNFP.featurize(
                self.structure, i) for i in range(len(self.structure))]
        except Exception:
            # counts crystal as invalid if fingerprint cannot be constructed.
            print('oops')
            self.valid = False
            self.comp_fp = None
            self.struct_fp = None
            return
        self.struct_fp = np.array(site_fps).mean(axis=0)

In [15]:
# collect sampled crystals in this batch.
batch_frac_coords.append(cur_frac_coords.detach().cpu())
batch_num_atoms.append(num_atoms.detach().cpu())
batch_atom_types.append(cur_atom_types.detach().cpu())
batch_lengths.append(lengths.detach().cpu())
batch_angles.append(angles.detach().cpu())

num_atoms2 = []
lengths2 = []
angles2 = []

# collect sampled crystals for this z.
frac_coords.append(torch.stack(batch_frac_coords, dim=0))
num_atoms2.append(torch.stack(batch_num_atoms, dim=0))
atom_types.append(torch.stack(batch_atom_types, dim=0))
lengths2.append(torch.stack(batch_lengths, dim=0))
angles2.append(torch.stack(batch_angles, dim=0))

all_frac_coords_stack = []
all_atom_types_stack = []
frac_coords = []
num_atoms = []
atom_types = []
lengths = []
angles = []
input_data_list = []

batch_all_frac_coords = []
batch_all_atom_types = []
batch_frac_coords, batch_num_atoms, batch_atom_types = [], [], []
batch_lengths, batch_angles = [], []

# Step 1: Create the dictionary with the concatenated tensors
crys_array_dict = {
    'frac_coords': frac_coords,
    'atom_types': atom_types,
    'lengths': lengths2,
    'angles': angles2
}

crys_array_dict

# annealed langevin dynamics.
for sigma in tqdm(self.sigmas, total=self.sigmas.size(0), disable=ld_kwargs.disable_bar):
    if sigma < ld_kwargs.min_sigma:
        break
    step_size = ld_kwargs.step_lr * (sigma / self.sigmas[-1]) ** 2

    for step in range(ld_kwargs.n_step_each):
        noise_cart = torch.randn_like(
            cur_frac_coords) * torch.sqrt(step_size * 2)
        pred_cart_coord_diff, pred_atom_types = self.decoder(
            z, cur_frac_coords, cur_atom_types, num_atoms, lengths, angles, gt_elements=gt_elements_input)
        cur_cart_coords = frac_to_cart_coords(
            cur_frac_coords, lengths, angles, num_atoms)
        pred_cart_coord_diff = pred_cart_coord_diff / sigma
        cur_cart_coords = cur_cart_coords + step_size * pred_cart_coord_diff + noise_cart
        cur_frac_coords = cart_to_frac_coords(
            cur_cart_coords, lengths, angles, num_atoms)

        if gt_atom_types is None:
            cur_atom_types = torch.argmax(pred_atom_types, dim=1) + 1
            print("current atom types are " + str(cur_atom_types))

        if ld_kwargs.save_traj:
            all_frac_coords.append(cur_frac_coords)
            all_pred_cart_coord_diff.append(
                step_size * pred_cart_coord_diff)
            all_noise_cart.append(noise_cart)
            all_atom_types.append(cur_atom_types)

output_dict = {'num_atoms': num_atoms, 'lengths': lengths, 'angles': angles,
            'frac_coords': cur_frac_coords, 'atom_types': cur_atom_types,
            'is_traj': False}

if ld_kwargs.save_traj:
    output_dict.update(dict(
        all_frac_coords=torch.stack(all_frac_coords, dim=0),
        all_atom_types=torch.stack(all_atom_types, dim=0),
        all_pred_cart_coord_diff=torch.stack(
            all_pred_cart_coord_diff, dim=0),
        all_noise_cart=torch.stack(all_noise_cart, dim=0),
        is_traj=True))